In [ ]:
import os
os.environ["HF_HOME"] = "/home/yandex/APDL2425a/group_12/gorodissky/.cache/huggingface"
print(f"HF_HOME set to:\t\t {os.environ['HF_HOME']}")

import torch
print(f"CUDA available: \t{torch.cuda.is_available()}")
print(f"Torch version: \t\t{torch.__version__}")
if torch.cuda.is_available():
    print(f"Number of CUDA devices\t {torch.cuda.device_count()}")
    print(f"CUDA device:\t\t {torch.cuda.get_device_name(torch.cuda.current_device())}")

In [ ]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, load_from_disk, disable_progress_bar
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os
from transformers import AutoModelForCausalLM, AutoModel, AutoTokenizer, AutoConfig, DataCollatorWithPadding
from sae.probes import eval_baseline, eval, create_dataset_baseline, create_datasets
from collections import defaultdict
import numpy as np

In [ ]:
# Plot length regression results for all models
model_to_colors = {
    "Qwen/Qwen2.5-7B-Instruct": "blue",
    "google/gemma-2-9b-it": "green",
    "mistralai/Ministral-8B-Instruct-2410": "red",
    "meta-llama/Llama-3.1-8B-Instruct": "orange",
}
dataset_name = "allenai/WildChat-1M"
task = "length"
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for model_name, color in model_to_colors.items():
    print(f"Evaluating model: {model_name}")

    layers , rel_errs, r2_scores = eval(
        model_name=model_name,
        task=task,
        dataset=dataset_name,
    )
    layers_ratio = [layer / max(layers) for layer in layers]
    # plot rel err
    axes[0].plot(layers_ratio, rel_errs, marker='o', color=color, label=model_name)
    axes[0].set_title(f"Predicting {task}")
    axes[0].set_xlabel("Layer (normalized)")
    axes[0].set_ylabel("Relative Error")
    axes[0].legend()
    axes[0].grid(True)

    # plot r2 scores
    axes[1].plot(layers_ratio, r2_scores, marker='o', color=color, label=model_name)
    axes[1].set_title(f"Predicting {task}")
    axes[1].set_xlabel("Layer (normalized)")
    axes[1].set_ylabel("R2 Score")
    axes[1].legend()
    axes[1].grid(True)

plt.tight_layout()
plt.show()